# Ordered Logistic Regression Results for Adoption Predictors Exploration with `mlcroissant`
This notebook provides a walkthrough for loading and exploring the [FAIR²](https://doi.org/10.71728/senscience.y7m0-f273) dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is defined by the following Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json`

In [ ]:
# Ensure mlcroissant is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Display dataset-level metadata
metadata = dataset.metadata
print(f"Dataset Title: {metadata.name}\n")
print("Description:")
print(metadata.description)
print("\nIdentifier:", metadata.identifier)
print("Authors:")
try:
    for author in metadata.author:
        print(f"  - {author}")
except Exception:
    print(metadata.author)
print("\nCollection timeframe:", getattr(metadata, 'dataCollectionTimeframe', None))

## 2. Data Overview
Review available record sets and their fields by their `@id`. Croissant organizes data in *Record Sets*, each describing a logical table or entity collection.

In [ ]:
# List all record sets in the dataset by their @id
print("Available Record Sets:")
record_sets = dataset.record_sets
for rs in record_sets:
    # Each record set has @id, name, etc.
    print(f"- @id: {rs.id} | name: {getattr(rs, 'name', '[no name]')}")
    if hasattr(rs, 'fields'):
        print("  Fields:")
        for field in rs.fields:
            print(f"    - @id: {field.id} | name: {getattr(field, 'name', '[no name]')}")
    print("")

# Save the list of record set @id values for extraction
record_set_ids = [rs.id for rs in record_sets]

## 3. Data Extraction
Load data from each record set using its `@id` into a DataFrame for analysis. Record set and field `@id` values can be referenced from the overview above.

In [ ]:
# Extract records from each record set into Pandas DataFrames using their @id
dataframes = {}
print("Extracting data for record sets:")
for rs_id in record_set_ids:
    print(f"- {rs_id}")
    records = list(dataset.records(record_set=rs_id))
    if records:  # Only save non-empty DataFrames
        dataframes[rs_id] = pd.DataFrame(records)
    else:
        print(f"  No records found for record set {rs_id}")

# Show columns from the first non-empty DataFrame
if dataframes:
    first_rs_id = next(iter(dataframes))
    print(f"\nColumns in record set {first_rs_id}:")
    print(dataframes[first_rs_id].columns.tolist())
    print("\nPreview of the data:")
    display(dataframes[first_rs_id].head())
else:
    print("No dataframes available (no records in any record set)")

## 4. Exploratory Data Analysis (EDA)
Apply data processing: filter records, normalize numeric fields, and group by key attributes using `@id` identifiers.

Below, select a numeric field and a grouping/categorical field to demonstrate typical EDA steps.

In [ ]:
import numpy as np

# If there's no data, skip further steps
if not dataframes:
    print("No record sets with data to analyze.")
else:
    # Choose the first record set with data
    rs_id = next(iter(dataframes))
    df = dataframes[rs_id]

    # Infer numeric field from DataFrame (example: select first float/int column)
    # For reproducibility, prefer known @id if available, else fallback to dtype
    numeric_fields = [col for col in df.columns if np.issubdtype(df[col].dtype, np.number)]
    if numeric_fields:
        numeric_field_id = numeric_fields[0]
    else:
        print("No numeric fields found in record set! EDA skipped.")
        numeric_field_id = None

    if numeric_field_id:
        threshold = df[numeric_field_id].dropna().mean()  # Example: use mean as threshold
        filtered_df = df[df[numeric_field_id] > threshold].copy()
        print(f"Filtered records (using {numeric_field_id} > {threshold:.2f}):")
        display(filtered_df.head())

        # Normalize the field
        filtered_df[f"{numeric_field_id}_normalized"] = (
            (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) /
            filtered_df[numeric_field_id].std()
        )
        print(f"\nNormalized '{numeric_field_id}' for filtered records:")
        display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Try grouping: use the first non-numeric field
        group_candidates = [col for col in df.columns if not np.issubdtype(df[col].dtype, np.number)]
        group_field_id = group_candidates[0] if group_candidates else None
        if group_field_id:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().to_frame()
            print(f"\nGrouped data by '{group_field_id}':")
            display(grouped_df.head())
        else:
            print("No suitable group field found.")

## 5. Visualization
Visualize data distributions or relationships between record set fields using `@id` references.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if dataframes and numeric_field_id:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field_id].dropna(), kde=True, bins=20)
    plt.title(f"Distribution of '{numeric_field_id}'")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

    if group_field_id:
        plt.figure(figsize=(10, 5))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
        plt.title(f"'{numeric_field_id}' grouped by '{group_field_id}'")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()

## 6. Conclusion
In this notebook, you have loaded the FAIR² dataset via its Croissant schema, explored its record sets and fields (referenced by their `@id`), and performed basic data analysis using the `mlcroissant` Python library.

Key findings, data distributions, and relationships between fields can be further explored by referencing the dataset schema and entity `@id`s for robust, reproducible analyses.

Continue exploring, filtering, or visualizing using the provided DataFrames and the rich metadata from the Croissant description!